In [ ]:
# 所有lod1城市对应的表名前缀
LOD1_CITIES = [
    "hamburg", "amsterdam", "dresden", "berlin", "rotterdam",
    "luxembourg", "sanfrancisco", "chicago", "boston", "washington",
    "newyork", "portland", "tokyo", "sapporo", "osaka", "kanazawa",
    "toronto", "vienna", "winnipeg", "zurich", "bologna",
    "paris", "lyon", "marseille"
]

# 1. Block指标

In [ ]:
import os
from dotenv import load_dotenv

import utils_z

In [ ]:
load_dotenv()

db_config = {
    "host": os.getenv("DB_HOST"),
    "database": os.getenv("DB_NAME"),
    "user": os.getenv("DB_USER"),
    "password": os.getenv("DB_PASSWORD"),
    "port": int(os.getenv("DB_PORT"))
}
conn = utils_z.get_conn(**db_config)

In [ ]:
import psycopg2
import pandas as pd
from psycopg2 import sql

In [ ]:
def add_metric_columns(conn, city):
    """为block表新增指标字段（若已存在则跳过）"""
    table = f"block.{city}_blocks"
    columns = {
        "elongation": "NUMERIC",
        "compactness": "NUMERIC",
        "building_count": "INTEGER",
        "bcr": "NUMERIC",
        "geom_is_valid": "BOOLEAN"
    }
    with conn.cursor() as cur:
        for col, dtype in columns.items():
            cur.execute(f"""
                ALTER TABLE {table}
                ADD COLUMN IF NOT EXISTS {col} {dtype};
            """)
    conn.commit()
    print(f"[{city}] 字段添加完成")

def compute_metrics(conn, city):
    """计算5个指标并写回block表"""
    table = f"block.{city}_blocks"
    bld_table = f"lod1.{city}_buildings_lod1"

    # Step 1: 计算几何指标（elongation, compactness, geom_is_valid）
    # 使用geography类型保证单位为米
    geom_sql = f"""
        UPDATE {table} b
        SET
            geom_is_valid = ST_IsValid(b.geom),
            compactness = CASE
                WHEN ST_Perimeter(b.geom::geography) > 0
                THEN (4 * PI() * ST_Area(b.geom::geography))
                     / (ST_Perimeter(b.geom::geography) ^ 2)
                ELSE NULL
            END,
            elongation = CASE
                WHEN ST_IsValid(b.geom)
                THEN (
                    SELECT
                        GREATEST(width, height) / NULLIF(LEAST(width, height), 0)
                    FROM (
                        SELECT
                            ST_Length(ST_LongestLine(
                                ST_OrientedEnvelope(b.geom),
                                ST_OrientedEnvelope(b.geom)
                            )::geography) AS height,
                            ST_Area(ST_OrientedEnvelope(b.geom)::geography)
                            / NULLIF(ST_Length(ST_LongestLine(
                                ST_OrientedEnvelope(b.geom),
                                ST_OrientedEnvelope(b.geom)
                            )::geography), 0) AS width
                    ) dims
                )
                ELSE NULL
            END
        WHERE b.geom IS NOT NULL;
    """

    # Step 2: 从建筑表聚合building_count和bcr
    bld_sql = f"""
        UPDATE {table} b
        SET
            building_count = agg.cnt,
            bcr = CASE
                WHEN b.area_m2 > 0
                THEN agg.total_footprint / b.area_m2
                ELSE NULL
            END
        FROM (
            SELECT
                block_id,
                COUNT(*) AS cnt,
                SUM(ST_Area(geom_2d::geography)) AS total_footprint
            FROM {bld_table}
            WHERE block_id IS NOT NULL
            GROUP BY block_id
        ) agg
        WHERE b.block_id = agg.block_id;
    """

    with conn.cursor() as cur:
        print(f"[{city}] 计算几何指标...")
        cur.execute(geom_sql)
        print(f"[{city}] 计算建筑指标...")
        cur.execute(bld_sql)
    conn.commit()
    print(f"[{city}] ✓ 完成")

In [ ]:
def run_all(conn, cities):
    for city in cities:
        try:
            add_metric_columns(conn, city)
            compute_metrics(conn, city)
        except Exception as e:
            conn.rollback()
            print(f"[{city}] ✗ 错误: {e}")

run_all(conn, LOD1_CITIES)